# RAG 성능을 높이는 LLM 파인튜닝


- LLM에 사용자가 가진 데이터셋을 추가로 학습시켜서 해당 데이터에 한해서 더 좋은 성능을 얻을 수 있도록 조정하는 과정을 파인튜닝이라고 합니다.
- 파인튜닝을 통해 다양한 문제를 풀 수 있지만, RAG의 성능을 높이기 위해서도 사용할 수 있습니다.


## RAFT 논문 살펴보기

- 미국 버클리 대학에서 작성한 "RAFT: Adapting Language Model to Domain Specific RAG"라는 논문이 있습니다.
- https://arxiv.org/pdf/2403.10131


### 네거티브 샘플

- 인공지능 분야에서 네거티브 샘플은 주로 모델이 찾고자 하는 대상과 반대되는 예시들을 의미합니다.
- 예를 들어 스팸 메일 분류기를 만든다고 해봅시다.
- 스팸 메일 분류기 입장에서는 정상 이메일이 네거티브 샘플에 해당합니다.
- 모델은 스팸 이메일을 찾아야 하므로 스팸 메일이 포지티브 샘플이라고 합니다.


네거티브 샘플 (장영실, 이순신, 이방원), 세종대왕 + 한글을 찾에한 조선의 왕은? <= 모델이 찾아야 할 것

- RAG에서 네거티브 샘플은 어떤 것이 있을까?
- LLM이 챗봇에 질문에 답변하려면 '세종대왕' 문서를 참고하야 하고, 나머지는 한글 창제와 관련된 이야기가 없다고 하면 이들문서가 네거티브 샘플입니다.
- 다시 말해 사용자 질의에 대한 정답이 있는 문서가 포지티브 샘플, 검색은 되었지만 정답이 포함되지 않은 문서들이 네거티브 샘플입니다.
- LLM은 해당 상황에서 연관되지 않은 문서의 내용은 무시하고, 연관된 문서의 내용만을 참고해 답변하는 능력이 길러져 있어야 합니다.
- 따라서 RAFT 논문에서는 이러한 상황에 대해 LLM이 충분히 학습할 수 있도록 학습데이터에 네거티브 샘플이 포함되도록 데이터셋을 구성하고 있습니다.


### 생각의 사슬

- 생각의 사슬 기법이란 LLM이 답변을 작성할 때 문제의 인과 관계에 대해 차근차근 풀어서 전개하다 보면, 정답에 더 잘 도달할 수 있다는 개념입니다.
- 문제-답 대신 문제-풀이-답 형태로 프롬프트를 구성하도록 하는 것입니다.


- RAFT 논문에서 사용한 생각의 사슬 프롬프트를 확인해봅시다.

**질문**: 오베로이 가문은 본사가 어느 도시에 있는 호텔 회사의 일부입니까?

**문맥**: [오베로이 가문은 호텔 사업으로 유명한 인도 가문입니다. 주로 오베로이 그룹을 통해 호텔 사업에 참여하고 있습니다...
이는 자카르타 시내 중심에 위치하고 있으며, 메가 쿠닝안 근처에 있으며, 자매 JW 메리어트 호텔 옆에 있습니다.
이는 리츠칼튼 호텔 회사에 의해 운영됩니다. 이 복합 단지는 호텔과 에어랑가 아파트로 구성된 두 개의 타워로 이루어져 있습니다...]
...
[오베로이 그룹은 본사가 델리에 있는 호텔 회사입니다.]

**지시사항**: 위의 질문, 문맥 및 답변을 고려하여 논리적 추론을 제공하십시오. 답변 형식은 다음과 같이 해주십시외 ##이유: {이유} ##답변: {답변}

**COT 답변**: ##이유: 문서에서 ##인용시작## 오베로이 가문은 호텔 사업으로 유명한 인도 가문입니다. 주로 오베로이 그룹을 통해 호텔 사업에 참여하고 있습니다. ##인용종료## 는 오베로이 가문이 오베로이 그룹에 참여하고 있음을 명시합니다. 그리고 문서에서 ##인용시작## 오베로이 그룹은 본사가 델리에 있는 호텔회사입니다. ##인용종료## 는 오베로이 그룹의 본사가 델리에 있음을 명시합니다. 따라서, 오베로이 가문은 본사가 델리에 있는 호텔 회사의 일부 입니다.

##답변: 델리


- RAFT 논문에서 사용한 프롬프트와 답변 형식입니다. 주요 입력으로 질문, 문맥, 지시사항 세가지가 존재합니다.
- 사용자가 질문을 입력하면 LLM은 주어진 문맥으로부터 질문에 대해 답변해야 합니다.
- 문맥은 사용자 질문으로부터 검색된 문서가 들어가는 위치입니다.
- 지시사항을 보면 사용자의 질문에 답변하기 전에 먼저 이유를 작성하라고 강제하고 있습니다.
- 답변을 작성하기 전에 중간 추론 과정을 강제하고, 모델은 이유를 작성하고 답변을 작성할 것이므로 이 방식은 생각의 사슬 프롬프트입니다.


- CoT 답변: 부분을 보면 LLM이 생성하게 될 출력입니다.
- 모델은 주어진 지시사항에 따라 답변전 "##이유:"에 근거를 작성해야 합니다.
- RAFT 논문에서는 이유를 작성할 때 '##인용시작##'과 '##인죵종료##'를 사용하여 실제 주어진 문맥에서 원문을 인용하도록 강제하고 있습니다.
- '##인용시작##'과 '##인용종료##' 사이에 있는 문장은 반드시 문맥에 주어진 원문에 존재하는 문장을 그대로 작성해야 합니다.


- 답변을 작성하기 전에 반드시 원문을 인용하도록 강제함으로써 주어진 문맥을 통해서만 답변하도록 유도합니다.
- 원문을 기반으로 강제하는것만으로도 RAG성능을 올릴 수 있습니다.


## 성능 향상을 위한 팁


### 답변 없음 데이터

- 사용자가 갖고 있는 문서들로는 답변할 수 없는 질문을 던지거나, 검색 성능이 좋지 않아서 사용자 질의를 고려한 검색 결과를 얻을 수 없는 상황에는, 모두 네거티브 샘플일 수 있습니다.
- LLM은 검색 문서에 없는 내용이라고 해서 모델이 스스로 답변할 경우, 할루시네이션이 발생하여 완전히 잘못된 답변이 생성될 여지가 있습니다.
- 따라서 모든 검색 문서가 네거티브 샘플일 경우, LLM이 '검색 결과를 찾을 수 없습니다'와 같은 답변을 하도록 학습시켜 자체적으로 답변하는 것을 막을 수 있습니다.


- 답변의 주요 내용이 검색 결과로 나온 문서 중 어떤 문서에서 나왔는지 남기도록 학습시키면 사용자에게 신뢰를 줄 수 있을 뿐만 아니라, 성능 측면에서도 모델이 잘못된 답변을 할 가능성도 줄일 수 있습니다.


## RAG 학습 데이터셋 살펴보기

### 학습 데이터 소개

- https://huggingface.co/datasets/iamjoon/klue-mrc-ko-rag-dataset


### 학습 데이터 탐색

[ch08_RAG_DATASET.ipynb](ch08_RAG_DATASET.ipynb)
